# Упражнение: метод k ближайших соседей (kNN)

Классификатор kNN работает в два этапа:

- Во время обучения классификатор получает обучающие данные и просто запоминает их.
- Во время тестирования kNN классифицирует каждое тестовое изображение: сравнивает его со всеми обучающими изображениями и присваивает ему метку по меткам k самых похожих обучающих примеров.
- Значение k подбирается с помощью кросс-валидации.

В этом упражнении вы реализуете эти шаги, разберётесь в базовом конвейере (pipeline) классификации изображений и в кросс-валидации, а также научитесь писать эффективный векторизованный код.

In [ ]:
# Настройка окружения (запустите эту ячейку первой).
# Откройте ноутбук из папки python/ задания, например: uv run jupyter lab
# При первом запуске скачается CIFAR-10 (~170 МБ): с сервера курса, а если он
# недоступен, с официального сайта. Источник можно задать явно переменной
# окружения DLCV_DATASET_SOURCE=racoon или official перед запуском Jupyter.
import os
import sys

if not os.path.isdir('dlcv'):
    raise FileNotFoundError(
        'В текущей папке нет пакета dlcv. Откройте ноутбук из папки python/ задания.')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from dlcv.data_utils import ensure_cifar10
ensure_cifar10()
print('Рабочая папка:', os.getcwd())

In [ ]:
# Подготовительный код для этого ноутбука.

import random
import numpy as np
from dlcv.data_utils import load_CIFAR10
import matplotlib.pyplot as plt

# Немного магии, чтобы графики matplotlib отображались прямо в ноутбуке,
# а не в отдельном окне.
%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # размер графиков по умолчанию
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# Ещё немного магии, чтобы ноутбук автоматически перезагружал внешние
# модули Python; подробнее см.
# http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

In [ ]:
# Загружаем исходные данные CIFAR-10.
cifar10_dir = 'dlcv/datasets/cifar-10-batches-py'

# Удаляем переменные, чтобы не загружать данные несколько раз
# (это может привести к нехватке памяти)
try:
   del X_train, y_train
   del X_test, y_test
   print('Ранее загруженные данные удалены.')
except:
   pass

X_train, y_train, X_test, y_test = load_CIFAR10(cifar10_dir)

# Для быстрой проверки выведем формы массивов обучающих и тестовых данных.
print('Форма обучающих данных: ', X_train.shape)
print('Форма обучающих меток: ', y_train.shape)
print('Форма тестовых данных: ', X_test.shape)
print('Форма тестовых меток: ', y_test.shape)

In [ ]:
# Визуализируем несколько примеров из набора данных.
# Показываем по нескольку обучающих изображений каждого класса.
classes = ['самолёт', 'автомобиль', 'птица', 'кошка', 'олень', 'собака', 'лягушка', 'лошадь', 'корабль', 'грузовик']
num_classes = len(classes)
samples_per_class = 7
for y, cls in enumerate(classes):
    idxs = np.flatnonzero(y_train == y)
    idxs = np.random.choice(idxs, samples_per_class, replace=False)
    for i, idx in enumerate(idxs):
        plt_idx = i * num_classes + y + 1
        plt.subplot(samples_per_class, num_classes, plt_idx)
        plt.imshow(X_train[idx].astype('uint8'))
        plt.axis('off')
        if i == 0:
            plt.title(cls)
plt.show()

In [ ]:
# Берём подвыборку данных, чтобы код в этом упражнении выполнялся быстрее
num_training = 5000
mask = list(range(num_training))
X_train = X_train[mask]
y_train = y_train[mask]

num_test = 500
mask = list(range(num_test))
X_test = X_test[mask]
y_test = y_test[mask]

# Разворачиваем каждое изображение в строку
X_train = np.reshape(X_train, (X_train.shape[0], -1))
X_test = np.reshape(X_test, (X_test.shape[0], -1))
print(X_train.shape, X_test.shape)

In [ ]:
from dlcv.classifiers import KNearestNeighbor

# Создаём экземпляр классификатора kNN.
# Напомним, что «обучение» классификатора kNN ничего не делает (no-op):
# классификатор просто запоминает данные и никак их больше не обрабатывает
classifier = KNearestNeighbor()
classifier.train(X_train, y_train)

Теперь классифицируем тестовые данные с помощью классификатора kNN. Напомним, что этот процесс можно разбить на два шага:

1. Сначала нужно вычислить расстояния между всеми тестовыми и всеми обучающими примерами.
2. Затем по этим расстояниям для каждого тестового примера находим k ближайших примеров, которые «голосуют» за его метку.

Начнём с вычисления матрицы расстояний между всеми обучающими и тестовыми примерами. Например, если есть **Ntr** обучающих и **Nte** тестовых примеров, на этом этапе должна получиться матрица размера **Nte x Ntr**, в которой элемент (i,j) — расстояние между i-м тестовым и j-м обучающим примером.

**Примечание: ни в одном из трёх способов вычисления расстояний, которые нужно реализовать в этом ноутбуке, нельзя использовать функцию np.linalg.norm() из numpy.**

Сначала откройте `dlcv/classifiers/k_nearest_neighbor.py` и реализуйте функцию `compute_distances_two_loops`, которая с помощью (очень неэффективного) двойного цикла по всем парам (тестовый, обучающий) примеров вычисляет матрицу расстояний поэлементно.

In [ ]:
# Откройте dlcv/classifiers/k_nearest_neighbor.py и реализуйте
# compute_distances_two_loops.

# Проверьте свою реализацию:
dists = classifier.compute_distances_two_loops(X_test)
print(dists.shape)

In [ ]:
# Матрицу расстояний можно визуализировать: каждая строка соответствует одному
# тестовому примеру и содержит его расстояния до обучающих примеров
plt.imshow(dists, interpolation='none')
plt.show()

**Вопрос 1**

Обратите внимание на структурные закономерности в матрице расстояний: некоторые строки или столбцы заметно ярче остальных. (При цветовой схеме по умолчанию чёрный цвет соответствует малым расстояниям, а белый — большим.)

- Какая особенность данных приводит к появлению отчётливо ярких строк?
- Чем вызваны яркие столбцы?

**Ваш ответ:** *впишите ответ здесь*


In [ ]:
# Теперь реализуйте функцию predict_labels и запустите код ниже.
# Используем k = 1 (то есть метод ближайшего соседа).
y_test_pred = classifier.predict_labels(dists, k=1)

# Вычисляем и выводим долю правильно предсказанных примеров
num_correct = np.sum(y_test_pred == y_test)
accuracy = float(num_correct) / num_test
print('Верно предсказано %d / %d => точность: %f' % (num_correct, num_test, accuracy))

Точность (accuracy, доля правильных ответов) должна получиться примерно `27%`. Теперь попробуем большее значение `k`, например `k = 5`:

In [ ]:
y_test_pred = classifier.predict_labels(dists, k=5)
num_correct = np.sum(y_test_pred == y_test)
accuracy = float(num_correct) / num_test
print('Верно предсказано %d / %d => точность: %f' % (num_correct, num_test, accuracy))

Результат должен получиться немного лучше, чем при `k = 1`.

**Вопрос 2**

Можно использовать и другие метрики расстояния, например расстояние L1.
Пусть $p_{ij}^{(k)}$ — значение пикселя в позиции $(i,j)$ изображения $I_k$.

Тогда среднее $\mu$ по всем пикселям всех изображений равно $$\mu=\frac{1}{nhw}\sum_{k=1}^n\sum_{i=1}^{h}\sum_{j=1}^{w}p_{ij}^{(k)}$$
а попиксельное среднее $\mu_{ij}$ по всем изображениям равно
$$\mu_{ij}=\frac{1}{n}\sum_{k=1}^np_{ij}^{(k)}.$$
Общее стандартное отклонение $\sigma$ и попиксельное стандартное отклонение $\sigma_{ij}$ определяются аналогично.

Какие из перечисленных ниже способов предобработки не изменят качество работы классификатора по методу ближайшего соседа с расстоянием L1? Укажите все подходящие варианты. Уточним: обучающие и тестовые примеры предобрабатываются одинаково.

1. Вычитание среднего $\mu$ ($\tilde{p}_{ij}^{(k)}=p_{ij}^{(k)}-\mu$.)
2. Вычитание попиксельного среднего $\mu_{ij}$  ($\tilde{p}_{ij}^{(k)}=p_{ij}^{(k)}-\mu_{ij}$.)
3. Вычитание среднего $\mu$ и деление на стандартное отклонение $\sigma$.
4. Вычитание попиксельного среднего $\mu_{ij}$ и деление на попиксельное стандартное отклонение $\sigma_{ij}$.
5. Поворот координатных осей данных, то есть поворот всех изображений на один и тот же угол. Пустые области, возникающие на изображении при повороте, заполняются одним и тем же значением пикселя, интерполяция не выполняется.

**Ваш ответ:** *впишите ответ здесь*

**Ваше объяснение:** *впишите объяснение здесь*


In [ ]:
# Теперь ускорим вычисление матрицы расстояний с помощью частичной векторизации
# с одним циклом. Реализуйте функцию compute_distances_one_loop и запустите
# код ниже:
dists_one = classifier.compute_distances_one_loop(X_test)

# Чтобы убедиться, что векторизованная реализация верна, проверим, что её
# результат совпадает с результатом наивной реализации. Решить, похожи ли две
# матрицы, можно многими способами; один из самых простых — норма Фробениуса.
# Если вы с ней ещё не встречались: норма Фробениуса разности двух матриц —
# это квадратный корень из суммы квадратов разностей всех элементов; иначе
# говоря, нужно развернуть матрицы в векторы и вычислить евклидово расстояние
# между ними.
difference = np.linalg.norm(dists - dists_one, ord='fro')
print('Разница для версии с одним циклом: %f' % (difference, ))
if difference < 0.001:
    print('Отлично! Матрицы расстояний совпадают')
else:
    print('Ой! Матрицы расстояний различаются')

In [ ]:
# Теперь реализуйте полностью векторизованную версию в
# compute_distances_no_loops и запустите код
dists_two = classifier.compute_distances_no_loops(X_test)

# проверяем, что матрица расстояний совпадает с вычисленной ранее:
difference = np.linalg.norm(dists - dists_two, ord='fro')
print('Разница для версии без циклов: %f' % (difference, ))
if difference < 0.001:
    print('Отлично! Матрицы расстояний совпадают')
else:
    print('Ой! Матрицы расстояний различаются')

In [ ]:
# Сравним, насколько быстро работают разные реализации
def time_function(f, *args):
    """
    Вызывает функцию f с аргументами args и возвращает время её выполнения (в секундах).
    """
    import time
    tic = time.time()
    f(*args)
    toc = time.time()
    return toc - tic

two_loop_time = time_function(classifier.compute_distances_two_loops, X_test)
print('Версия с двумя циклами: %f с' % two_loop_time)

one_loop_time = time_function(classifier.compute_distances_one_loop, X_test)
print('Версия с одним циклом: %f с' % one_loop_time)

no_loop_time = time_function(classifier.compute_distances_no_loops, X_test)
print('Версия без циклов: %f с' % no_loop_time)

# Полностью векторизованная реализация должна работать значительно быстрее!

# ПРИМЕЧАНИЕ: в зависимости от компьютера при переходе от двух циклов к одному
# ускорения может не быть, а может даже наблюдаться замедление.

### Кросс-валидация

Классификатор kNN реализован, но значение k = 5 было выбрано произвольно. Теперь определим наилучшее значение этого гиперпараметра с помощью кросс-валидации.

In [ ]:
num_folds = 5
k_choices = [1, 3, 5, 8, 10, 12, 15, 20, 50, 100]

X_train_folds = []
y_train_folds = []
################################################################################
# TODO:                                                                        #
# Разбейте обучающие данные на фолды. После разбиения X_train_folds и          #
# y_train_folds должны быть списками длины num_folds, причём                   #
# y_train_folds[i] — вектор меток для объектов из X_train_folds[i].            #
# Подсказка: посмотрите функцию numpy array_split.                             #
################################################################################


# Словарь со значениями точности для разных k, полученными при
# кросс-валидации. После кросс-валидации k_to_accuracies[k] должен быть
# списком длины num_folds со значениями точности, полученными при данном k.
k_to_accuracies = {}


################################################################################
# TODO:                                                                        #
# Выполните кросс-валидацию по фолдам (k-fold), чтобы найти наилучшее          #
# значение k. Для каждого возможного значения k запустите алгоритм             #
# k ближайших соседей num_folds раз: каждый раз используйте все фолды, кроме   #
# одного, как обучающие данные, а оставшийся фолд — как валидационную          #
# выборку. Сохраните значения точности для всех фолдов и всех значений k       #
# в словаре k_to_accuracies.                                                   #
################################################################################


# Выводим полученные значения точности
for k in sorted(k_to_accuracies):
    for accuracy in k_to_accuracies[k]:
        print('k = %d, точность = %f' % (k, accuracy))

In [ ]:
# отображаем исходные наблюдения
for k in k_choices:
    accuracies = k_to_accuracies[k]
    plt.scatter([k] * len(accuracies), accuracies)

# строим линию тренда с планками погрешностей, соответствующими
# стандартному отклонению
accuracies_mean = np.array([np.mean(v) for k,v in sorted(k_to_accuracies.items())])
accuracies_std = np.array([np.std(v) for k,v in sorted(k_to_accuracies.items())])
plt.errorbar(k_choices, accuracies_mean, yerr=accuracies_std)
plt.title('Кросс-валидация по k')
plt.xlabel('k')
plt.ylabel('Точность на кросс-валидации')
plt.show()

In [ ]:
# По результатам кросс-валидации выше выберите наилучшее значение k,
# заново обучите классификатор на всех обучающих данных и проверьте его на
# тестовых данных. Точность на тестовых данных должна получиться около 28%.
best_k = 1

classifier = KNearestNeighbor()
classifier.train(X_train, y_train)
y_test_pred = classifier.predict(X_test, k=best_k)

# Вычисляем и выводим точность
num_correct = np.sum(y_test_pred == y_test)
accuracy = float(num_correct) / num_test
print('Верно предсказано %d / %d => точность: %f' % (num_correct, num_test, accuracy))

**Вопрос 3**

Какие из следующих утверждений о методе $k$ ближайших соседей ($k$-NN) верны в задаче классификации при любом $k$? Укажите все подходящие варианты.
1. Граница решений классификатора k-NN линейна.
2. Ошибка на обучающей выборке у 1-NN всегда меньше или равна ошибке у 5-NN (считайте, что в обучающей выборке нет одинаковых примеров с разными метками).
3. Ошибка на тестовой выборке у 1-NN всегда меньше, чем у 5-NN.
4. Время, необходимое для классификации тестового примера классификатором k-NN, растёт с увеличением размера обучающей выборки.
5. Ни одно из перечисленных.

**Ваш ответ:** *впишите ответ здесь*

**Ваше объяснение:** *впишите объяснение здесь*
